# 12. Paoay: apply the fixed crop model

**One task:** apply notebook 06’s saved classifier to notebook 11’s verified, ordered feature matrix. Save crop-only predictions and probabilities on the original grid.

No retraining, imagery downloads, Dynamic World fusion or local accuracy scoring occurs here. The three-crop model must choose among carrot, cabbage and potato, even on other cover. Keep this raw output separate from the final combined map. See the [workflow guide](../../docs/paoay_workflow.md).


## 1. Setup

Finish notebooks 06 and 08–11. The model is the exact package bound to notebook 11’s features; there is no new model-selection setting. Its local fitted artifact is loaded only after checksum/schema verification and is never overwritten.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import predictions
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

chunk_size = 4096  # Computational batch size, not a scientific threshold.
show_preview = True
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
feature_pointer = paoay_root / "features/feature_stage.json"
prediction_root = paoay_root / "predictions"
prediction_pointer = prediction_root / "prediction_stage.json"
write_stage_pointer(prediction_pointer, status="pending", reason="Notebook 12 has not finished the current prediction run")
stages = ["Verify ready features and model", "Apply fixed crop model", "Review prediction coverage", "Preview and verify outputs", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "12_paoay_predictions", stages, report_folder.parent / "progress", {"chunk_size": chunk_size},
)


## 2. Verify ready features and model

Pending/empty features stop this notebook. Dates, grid, feature order and the feature-bound model are verified before classification. The fitted classifier’s own class order controls probability mapping; it need not match the output class-code order.


In [ ]:
pipeline_progress.begin_stage("Verify ready features and model")
feature_handoff = resolve_stage_pointer(feature_pointer)
request = predictions.prepare_request(feature_handoff, prediction_root)
print("Verified feature handoff:", feature_handoff)
print("Feature-bound model:", request["model_manifest"])
print("Prediction folder:", request["folder"])
pipeline_progress.end_stage()


## 3. Apply the fixed model

Use one progress counter for inference batches. No feature rows are removed using broad-cover guidance or a new confidence threshold. Save all three probabilities in labelled bands, the top probability and the top-two margin.

Ties retain the classifier’s normal first-class decision and have a zero margin. These are model scores, not measured accuracy or calibrated local correctness probabilities.


In [ ]:
pipeline_progress.begin_stage("Apply fixed crop model")
def inference_progress(items):
    return pipeline_progress.track(items, "Apply saved crop model", unit="batch")
prediction_handoff = predictions.build(request, chunk_size=chunk_size, iterator=inference_progress)
bundle = predictions.load_prediction_bundle(prediction_handoff)
pipeline_progress.end_stage()


## 4. Review coverage and predicted areas

Only pixels with ready features receive a crop code. Other pixels stay nodata (0), not “other crop.” Percentages below use predicted pixels as the denominator; areas are crop-model predictions, not verified planted areas.


In [ ]:
pipeline_progress.begin_stage("Review prediction coverage")
show_table([{**row, "code": int(row["code"]), "pixels": int(row["pixels"]), "area_ha": float(row["area_ha"]),
             "percent_of_predicted_pixels": float(row["percent_of_predicted_pixels"])} for row in bundle["summary"]],
           title="Crop-only predicted areas")
manifest = bundle["manifest"]
print("ROI pixels:", manifest["roi_pixels"])
print("Predicted pixels:", manifest["row_count"])
print("ROI pixels without valid crop inputs:", manifest["roi_pixels"] - manifest["row_count"])
print("Tied top probabilities:", manifest["tied_pixels"])
print("Classifier class order:", ", ".join(manifest["classifier_class_order"]))
print("Class codes:", "; ".join(f"{name} = {code}" for name, code in manifest["class_codes"].items()))
pipeline_progress.end_stage()


## 5. Preview and publish the handoff

The preview is crop-only. Forest, built or bare surfaces may still receive a crop label from this restricted classifier. Notebook 13 will assemble broad-cover guidance alongside these preserved raw outputs.


In [ ]:
pipeline_progress.begin_stage("Preview and verify outputs")
if show_preview:
    from IPython.display import display
    import matplotlib.pyplot as plt
    figure = predictions.plot_predictions(bundle)
    preview = prediction_root / "previews" / (manifest["bundle_id"] + ".png")
    preview.parent.mkdir(parents=True, exist_ok=True)
    if not preview.exists():
        figure.savefig(preview, dpi=150)
    display(figure)
    plt.close(figure)
    print("Crop-only preview:", preview)
predictions.load_prediction_bundle(prediction_handoff)
write_stage_pointer(prediction_pointer, prediction_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

The fixed-calendar period still has unverified crop growth-stage alignment. These predictions do not establish local accuracy or independently confirm the KMZ notes.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified prediction handoff:", prediction_handoff)
print("Current prediction pointer:", prediction_pointer)
print("Saved model preserved; no retraining or accuracy evaluation performed.")
print("Next: notebook 13 will assemble the combined map in a separate batch.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
